In [5]:
import os
import cv2
import pandas as pd
import numpy as np
import gradio as gr
import kagglehub
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [6]:
print("🛡️ Fetching Deepfake Forensic Dataset...")
df_path = kagglehub.dataset_download("payaldhokane/deepfake-and-synthetic-media-detection-dataset")
csv_file = [f for f in os.listdir(df_path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(df_path, csv_file))

feature_names = ['visual_artifacts_score', 'lighting_inconsistency_score', 'compression_level']
df['Is_Deepfake'] = (df['visual_artifacts_score'] > 0.45).astype(int)

🛡️ Fetching Deepfake Forensic Dataset...
Using Colab cache for faster access to the 'deepfake-and-synthetic-media-detection-dataset' dataset.


In [7]:
X = df[feature_names].values
y = df['Is_Deepfake'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("⚙️ Training Simple KNN Classifier...")
knn_model = KNeighborsClassifier(n_neighbors=7, weights='distance')
knn_model.fit(X_scaled, y)
print("🎉 Detector Ready!")

⚙️ Training Simple KNN Classifier...
🎉 Detector Ready!


In [8]:
def detect_deepfake(input_image):
    if input_image is None:
        return "⚠️ Please upload an image first!"

    gray = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)

    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    artifacts_score = min(1.0, max(0.0, 1.0 - (laplacian_var / 500.0)))

    lighting_score = np.std(gray) / 255.0

    edges = cv2.Canny(gray, 100, 200)
    compression_score = np.mean(edges) / 255.0

    features = np.array([[artifacts_score, lighting_score, compression_score]])
    features_scaled = scaler.transform(features)

    prediction = knn_model.predict(features_scaled)[0]
    probabilities = knn_model.predict_proba(features_scaled)[0]
    confidence = probabilities[prediction] * 100

    if prediction == 1:
        return f"🚨 FAKE / DEEPFAKE Detected! (Confidence: {confidence:.1f}%)"
    else:
        return f"✅ REAL / AUTHENTIC Image Verified. (Confidence: {confidence:.1f}%)"

In [9]:
interface = gr.Interface(
    fn=detect_deepfake,
    inputs=gr.Image(label="📸 Upload Image File"),
    outputs=gr.Textbox(label="🔍 Result Verdict", lines=2),
    title="🛡️ Simple Deepfake Image Detector",
    description="Drop any picture below and click submit. The AI model checks its structural details against the training data to tell you if it's Real or Fake.",
    allow_flagging="never"
)

interface.launch()

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aefc8e86695169f531.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
